In [1]:
import os
import pandas as pd
from datasets import load_dataset

# 1. Load the dataset from Hugging Face
# 'sms_spam' is the standard ID for this dataset
dataset = load_dataset("cybersectony/PhishingEmailDetectionv2.0", split="train")

# 2. Convert to a Pandas DataFrame
df = pd.DataFrame(dataset)

# 3. Create the 'data' directory if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')

# 4. Save to CSV
# Using utf-8 here ensures your local file won't have the error from before
df.to_csv('data/email_spam.csv', index=False, encoding='utf-8')

print("Data successfully saved to data/email_spam.csv!")
print(df.head())

c:\Users\adars\.conda\envs\college\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 60000/60000 [00:00<00:00, 1579111.36 examples/s]


Data successfully saved to data/email_spam.csv!
                                             content  label
0                           https://www.galvnews.com      2
1  https://filedn.com/l0kwbcnxzlkqbihn1fmjrk4/fee...      3
2  http://www.www--wellsfargo--com--tj49329d48d6c...      3
3                      http://www.sleamcommulnity.ru      3
4                         http://www.chihabidine.com      3


In [2]:
df.shape

(120000, 2)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   content  120000 non-null  object
 1   label    120000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.8+ MB


In [ ]:
# import pandas as pd

# # 1. Load your newly saved CSV
# df = pd.read_csv('data/email_spam.csv')

# # 2. Map the numeric labels to human-readable text
# # This makes it much easier to verify your data later
# label_map = {
#     0: 'legitimate_email',
#     1: 'phishing_email',
#     2: 'legitimate_url',
#     3: 'phishing_url'
# }
# df['label_name'] = df['label'].map(label_map)

# # 3. Create a boolean 'is_spam' column (True for phishing, False for legitimate)
# # This works for both emails and URLs
# df['is_phishing'] = df['label'].isin([1, 3])

# # 4. Split the data based on the label categories
# # Labels 2 and 3 are specifically for URLs
# df_urls = df[df['label'].isin([2, 3])].copy()
# df_emails = df[df['label'].isin([0, 1])].copy()

# # 5. Save them to separate files
# df_urls.to_csv('data/urls_only.csv', index=False)
# df_emails.to_csv('data/emails_only.csv', index=False)

# print(f"Splitting complete!")
# print(f"URLs saved: {len(df_urls)} rows")
# print(f"Emails saved: {len(df_emails)} rows")

Splitting complete!
URLs saved: 106507 rows
Emails saved: 13493 rows


In [1]:
import pandas as pd 
df_emails = pd.read_csv('data/emails_only.csv')

In [2]:
df_emails.head()

,content,label,label_name,is_phishing
0,empty,1,phishing_email,True
1,revised - transitional steering committee meet...,0,legitimate_email,False
2,high desert linguistics society / students cal...,0,legitimate_email,False
3,the no . 1 source for software superstore . ge...,1,phishing_email,True
4,URL: http://boingboing.net/#85516333\nDate: No...,0,legitimate_email,False


In [1]:
import pandas as pd
import json
import os
import random
from datetime import datetime
import ollama

# ---------------- CONFIG ---------------- #

MODEL_NAME = "gpt-oss:20b-cloud"
GEN_SIZE = 100

TIMESTAMP = datetime.now().strftime("%d_%m_%Hh")
FOLDER_NAME = f"email_sim_data_{TIMESTAMP}_GEN_{GEN_SIZE}"
ERROR_FOLDER = os.path.join(FOLDER_NAME, "broken_pairs")

# --------------- LOAD DATA --------------- #

df = pd.read_csv("data/emails_only.csv")

# keep only required columns
df = df[["content", "label_name"]]

samples = df.sample(GEN_SIZE)

# --------------- FOLDERS ----------------- #

os.makedirs(FOLDER_NAME, exist_ok=True)
os.makedirs(ERROR_FOLDER, exist_ok=True)

print(f"Starting email generation in folder: {FOLDER_NAME}")


# -------- FEW SHOT EXAMPLE POOL ---------- #

FEW_SHOT_POOL = [

"""
Example (Corporate IT Notification vs Microsoft Phishing):
{
"ham": {
"subject": "VPN Maintenance Window Tonight",
"body": "Dear Team,\n\nThe company VPN gateway will undergo scheduled maintenance tonight from 11:30 PM to 1:00 AM IST. During this time remote access may be temporarily unavailable.\n\nRegards,\nIT Infrastructure Team"
},
"spam": {
"subject": "Urgent: Microsoft Account Security Alert",
"body": "Your Microsoft 365 account has been flagged for suspicious login attempts. Failure to verify ownership within 2 hours will lead to account suspension.\n\nVerify now: <URL>\n\nMicrosoft Security Team"
}
}
""",

"""
Example (University Notice vs Scholarship Scam):
{
"ham": {
"subject": "Reminder: Semester Project Submission",
"body": "Dear Students,\n\nThis is a reminder that the final semester project report submission deadline is March 5th. Please upload your documents through the student portal before the deadline.\n\nAcademic Office"
},
"spam": {
"subject": "Government Education Grant Approved",
"body": "Congratulations!\n\nYour profile has been shortlisted for the National Student Financial Aid Grant worth Rs 75,000.\n\nSubmit bank verification immediately to release funds: <URL>\n\nHigher Education Support Program"
}
}
""",

"""
Example (E-commerce Order vs Delivery Phishing):
{
"ham": {
"subject": "Your Amazon Order Has Shipped",
"body": "Hello,\n\nYour order containing 'USB-C Charging Cable' has been shipped and is expected to arrive tomorrow.\n\nTrack your package through your Amazon account.\n\nThank you for shopping with us."
},
"spam": {
"subject": "Delivery Failed – Address Confirmation Required",
"body": "Dear Customer,\n\nYour package could not be delivered due to incomplete address information.\n\nUpdate delivery details immediately to avoid return to sender: <URL>"
}
}
""",

"""
Example (Company HR Update vs Fake Job Offer):
{
"ham": {
"subject": "Updated Leave Policy for FY 2026",
"body": "Dear Employees,\n\nThe HR department has updated the leave policy effective April 2026. Please review the changes in the employee portal.\n\nRegards,\nHuman Resources"
},
"spam": {
"subject": "Remote Job Opportunity – Immediate Hiring",
"body": "Hello Candidate,\n\nWe found your profile suitable for our remote data entry project. Earn up to Rs 45,000 monthly working from home.\n\nRegister here to schedule HR interview: <URL>"
}
}
""",

"""
Example (Bank Notification vs Account Suspension Phishing):
{
"ham": {
"subject": "Transaction Alert for Your Bank Account",
"body": "Dear Customer,\n\nA debit transaction of Rs 2,350 was made using your debit card ending in 4587.\n\nIf you did not perform this transaction please contact the bank immediately."
},
"spam": {
"subject": "Your Bank Account Will Be Suspended",
"body": "Important Notice,\n\nDue to incomplete KYC verification your bank account will be permanently restricted within 12 hours.\n\nComplete verification immediately: <URL>"
}
}
"""
]


# ------------ GENERATION LOOP ------------ #

for i, (idx, row) in enumerate(samples.iterrows(), 1):

    original_text = row["content"]
    original_label = row["label_name"]

    selected_examples = random.sample(FEW_SHOT_POOL, 3)
    examples_string = "\n\n".join(selected_examples)

    prompt = f"""
You are an expert cybersecurity dataset generator.

Your task is to generate a **realistic pair of email messages**:
- one legitimate email (ham)
- one phishing email (spam)

CRITICAL RULES:

1. DIVERSITY
Use different real-world scenarios such as:
- corporate IT alerts
- bank notifications
- delivery updates
- university emails
- HR announcements
- password reset notifications
- cloud service alerts
- fake invoices
- crypto scams
- fake government refunds
- fake security alerts

2. PHISHING STYLE
Phishing emails should use:
- urgency
- authority impersonation
- financial rewards
- account suspension threats
- verification requests

3. LEGITIMATE EMAIL STYLE
Ham emails should resemble:
- internal corporate emails
- service confirmations
- newsletters
- meeting reminders
- support responses
- order confirmations

4. LINKS
Never generate real URLs.
Replace every link with the exact token:
<URL>

5. FORMAT
Return ONLY valid JSON.

Example format:

{{
"ham": {{
"subject": "...",
"body": "..."
}},
"spam": {{
"subject": "...",
"body": "..."
}}
}}

6. LANGUAGE
Use natural English or Hinglish but ONLY Latin alphabet.

--- FEW SHOT EXAMPLES ---
{examples_string}
-------------------------

Baseline example from dataset labeled '{original_label}':
"{original_text}"

Use the baseline only for inspiration of email length and style.
Generate a completely new scenario.
"""

    try:

        response = ollama.generate(
            model=MODEL_NAME,
            prompt=prompt,
            format="json"
        )

        raw_response = response["response"]

        data = json.loads(raw_response)

        file_path = os.path.join(FOLDER_NAME, f"pair{i}.json")

        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)

        print(f"✅ Saved: pair{i}.json")

    except Exception as e:

        error_file_path = os.path.join(ERROR_FOLDER, f"broken_pair{i}.txt")

        with open(error_file_path, "w", encoding="utf-8") as f:
            f.write(f"Error: {str(e)}\n\n")
            f.write("--- RAW RESPONSE ---\n")
            f.write(raw_response)

        print(f"❌ Broken: pair{i}.json")


print(f"\nGeneration complete. Check '{FOLDER_NAME}' for results.")

Starting email generation in folder: email_sim_data_05_03_13h_GEN_100
✅ Saved: pair1.json
✅ Saved: pair2.json
✅ Saved: pair3.json
✅ Saved: pair4.json
✅ Saved: pair5.json
✅ Saved: pair6.json
✅ Saved: pair7.json
✅ Saved: pair8.json
✅ Saved: pair9.json
✅ Saved: pair10.json
✅ Saved: pair11.json
✅ Saved: pair12.json
✅ Saved: pair13.json
❌ Broken: pair14.json
✅ Saved: pair15.json
✅ Saved: pair16.json
✅ Saved: pair17.json
✅ Saved: pair18.json
✅ Saved: pair19.json
✅ Saved: pair20.json
✅ Saved: pair21.json
✅ Saved: pair22.json
❌ Broken: pair23.json
✅ Saved: pair24.json
✅ Saved: pair25.json
✅ Saved: pair26.json
✅ Saved: pair27.json
❌ Broken: pair28.json
✅ Saved: pair29.json
✅ Saved: pair30.json
✅ Saved: pair31.json
❌ Broken: pair32.json
✅ Saved: pair33.json
✅ Saved: pair34.json
✅ Saved: pair35.json
✅ Saved: pair36.json
✅ Saved: pair37.json
✅ Saved: pair38.json
✅ Saved: pair39.json
✅ Saved: pair40.json
✅ Saved: pair41.json
✅ Saved: pair42.json
✅ Saved: pair43.json
✅ Saved: pair44.json
✅ Saved: pa